# Four-species immune dataset: prepare annotated h5ad files

This notebook prepares Human, Mouse, Pig, and rhesus macaque immune-cell matrices from **GSE186158** and writes:

- `processed_h5ad/human.h5ad`
- `processed_h5ad/mouse.h5ad`
- `processed_h5ad/pig.h5ad`
- `processed_h5ad/macaM.h5ad`

Cell types are stored in `adata.obs['cell_type']`. Only cells with a usable annotation are retained.

## Data source

- GEO Series: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE186158
- BioProject: PRJNA772751
- SRA: SRP342163
- Processed GEO archive: `GSE186158_RAW.tar`

The workflow uses these four processed 10x archives:

- `GSM5639494_Mouse.tar.gz`
- `GSM5639496_Pig.tar.gz`
- `GSM5639497_Monkey.tar.gz`
- `GSM5639498_Human.tar.gz`

Each archive contains `barcodes.tsv.gz`, `features.tsv.gz`, and `matrix.mtx.gz`.

In [ ]:
from pathlib import Path
import subprocess
import sys

VIGNETTE_DIR = Path.cwd().resolve()
if VIGNETTE_DIR.name != '4_species_immune':
    VIGNETTE_DIR = Path(r'D:\111icde_addition_experiments\MetaGeneFormer\Vignettes\4_species_immune')

DATA_DIR = VIGNETTE_DIR / 'data'
OUTPUT_DIR = VIGNETTE_DIR / 'processed_h5ad'
ANNOTATION_CSV = DATA_DIR / 'four_species_immune_barcode_cell_type.csv'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Vignette:', VIGNETTE_DIR)
print('Annotation exists:', ANNOTATION_CSV.exists())

## Option A: use the existing local archives without moving the large data

The conversion script can read directly from the existing directory. This is the recommended option for the current workspace.

In [ ]:
INPUT_DIR = Path(r'D:\111icde_addition_experiments\4_species_immune')
required_archives = [
    INPUT_DIR / 'GSM5639494_Mouse.tar.gz',
    INPUT_DIR / 'GSM5639496_Pig.tar.gz',
    INPUT_DIR / 'GSM5639497_Monkey.tar.gz',
    INPUT_DIR / 'GSM5639498_Human.tar.gz',
]
for path in required_archives:
    print(path.name, 'OK' if path.exists() else 'MISSING')

## Option B: download from GEO into this vignette's `data` directory

The GEO archive is approximately 466 MB. The helper downloads the full archive temporarily, extracts only the four required GSM archives, and deletes the temporary full archive afterward. Skip this cell when using Option A.

In [ ]:
# subprocess.run([
#     sys.executable,
#     str(VIGNETTE_DIR / 'download_four_species_immune_data.py'),
#     '--data-dir', str(DATA_DIR),
# ], check=True)
# INPUT_DIR = DATA_DIR

## Annotation matching policy

The correct annotation file is `four_species_immune_barcode_cell_type.csv`. The similarly named gastric-antrum CSV has zero matching cells and must not be used.

The immune CSV contains historical species-prefix inconsistencies. To avoid assigning a wrong label, the script uses:

1. exact `species_barcode` matching first;
2. raw-barcode fallback only when all matching CSV records agree on one cell type;
3. removal of ambiguous or unannotated cells.

## Convert to QC-filtered h5ad

Defaults: at least 200 detected genes per cell, mitochondrial percentage at most 20%, and genes detected in at least 3 retained cells. Raw counts remain sparse and unnormalized in `adata.X`.

In [ ]:
subprocess.run([
    sys.executable,
    str(VIGNETTE_DIR / 'prepare_four_species_immune_h5ad.py'),
    '--input-dir', str(INPUT_DIR),
    '--annotation-csv', str(ANNOTATION_CSV),
    '--output-dir', str(OUTPUT_DIR),
    '--min-genes', '200',
    '--min-cells', '3',
    '--max-pct-mt', '20',
], check=True)

## Validate output

In [ ]:
import anndata as ad

for species in ('human', 'mouse', 'pig', 'macaM'):
    path = OUTPUT_DIR / f'{species}.h5ad'
    adata = ad.read_h5ad(path, backed='r')
    assert adata.obs_names.is_unique
    assert 'cell_type' in adata.obs.columns
    assert not adata.obs['cell_type'].isna().any()
    print(species, adata.shape, adata.obs['cell_type'].value_counts().to_dict())
    adata.file.close()